In [ ]:
library(tidyverse)
library(bigrquery)

#-- Extract basic person information (DOB and sex at birth)
person_25694592_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251014",  # Comment out this line if you want the export to always overwrite.
  "person_25694592",
  "person_25694592_*.csv")
message(str_glue('The data will be written to {person_25694592_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_25694592_person_df <- read_bq_export_from_workspace_bucket(person_25694592_path)
person_df <- dataset_25694592_person_df %>%
  select(person_id, date_of_birth, sex_at_birth) %>%
  mutate(
    sex_at_birth = case_when(
      !sex_at_birth %in% c("Female", "Male") ~ NA,
      TRUE ~ sex_at_birth
    )
  )



In [ ]:
#-- Extract basic SDOH information (income, education)
survey_25694592_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251014",  # Comment out this line if you want the export to always overwrite.
  "survey_25694592",
  "survey_25694592_*.csv")
message(str_glue('The data will be written to {survey_25694592_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_25694592_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_25694592_survey_df <- read_bq_export_from_workspace_bucket(survey_25694592_path)


In [ ]:
pheno <- dataset_25694592_survey_df %>%
  filter(question %in% c("Smoking: Smoke Frequency", "Income: Annual Income", "Education Level: Highest Grade", "Alcohol: Drink Frequency Past Year", "Insurance: Health Insurance", "In the last month, how often have you felt nervous and \"stressed\"?",
                        "Alcohol: Average Daily Drink Count", "Health Insurance: Health Insurance Type"))

In [ ]:
smoking <- pheno %>%
  filter(question == "Smoking: Smoke Frequency") %>%
  select(person_id, answer) %>%
  rename(Smoking_Frequency = answer)
education <- pheno %>%
  filter(question == "Education Level: Highest Grade") %>%
  select(person_id, answer) %>%
  rename(Education_level = answer)
alcohol <- pheno %>%
  filter(question == "Alcohol: Drink Frequency Past Year") %>%
  select(person_id, answer) %>%
  rename(Drink_Frequency = answer)
alc_amount <- pheno %>%
    filter(question == "Alcohol: Average Daily Drink Count") %>%
    select(person_id, answer) %>%
    rename(Drink_Amount = answer)
income <- pheno %>%
  filter(question == "Income: Annual Income") %>%
  select(person_id, answer) %>%
  rename(Income = answer)
stress <- pheno %>%
  filter(question == "In the last month, how often have you felt nervous and \"stressed\"?") %>%
  select(person_id, answer) %>%
  rename(Stress = answer)
insurance <- pheno %>%
    filter(question == "Insurance: Health Insurance") %>%
    select(person_id, answer) %>%
    rename(Insurance = answer)
pheno_all <- smoking %>%
  full_join(education, by = "person_id") %>%
  full_join(alcohol, by = "person_id") %>%
  full_join(alc_amount, by = "person_id") %>%
  full_join(income, by = "person_id") %>%
  full_join(stress, by = "person_id") %>%
  full_join(insurance, by = "person_id")


In [ ]:
#-- Clean phenotypes
final <- pheno_all %>%
  mutate(
    Smoking_Frequency = case_when(
      Smoking_Frequency == "Smoke Frequency: Every Day" ~ "Every Day",
      Smoking_Frequency == "Smoke Frequency: Not At All" ~ "Not At All",
      Smoking_Frequency == "Smoke Frequency: Some Days" ~ "Some Days",
      TRUE ~ NA
    ),
    Education_level = case_when(
      Education_level == "Highest Grade: Twelve Or GED" ~ "Twelve Or GED",
      Education_level %in% c("Highest Grade: One Through Four", 
                            "Highest Grade: Five Through Eight",
                            "Highest Grade: Nine Through Eleven",
                            "Highest Grade: Never Attended") ~ "Less than a high school degree",
      Education_level == "Highest Grade: College One to Three" ~ "College Undergraduate",
      Education_level == "Highest Grade: Advanced Degree" ~ "College Postgraduate",
      TRUE ~ NA
    ),
    Drink_Frequency = case_when(
      Drink_Frequency == "Drink Frequency Past Year: 2 to 3 Per Week" ~ "2-3 Per Week",
      Drink_Frequency == "Drink Frequency Past Year: 2 to 4 Per Month" ~ "2-4 Per Month",
      Drink_Frequency == "Drink Frequency Past Year: Monthly Or Less" ~ "Monthly Or Less",
      Drink_Frequency == "Drink Frequency Past Year: Never" ~ "Never",
      Drink_Frequency == "Drink Frequency Past Year: 4 or More Per Week" ~ ">=4 Per Week",
      TRUE ~ NA
    ),
    Income = case_when(
      Income == "Annual Income: 10k 25k" ~ "10-25k",
      Income == "Annual Income: 25k 35k" ~ "25-35k",
      Income == "Annual Income: 35k 50k" ~ "35-50k",
      Income == "Annual Income: 50k 75k" ~ "50-75k",
      Income == "Annual Income: 75k 100k" ~ "75-100k",
      Income == "Annual Income: 100k 150k" ~ "100-150k",
      Income == "Annual Income: 150k 200k" ~ "150-200k",
      Income == "Annual Income: more 200k" ~ ">= 200k",
      TRUE ~ NA
    ),
    Stress = case_when(
      Stress == "PMI: Skip" ~ NA,
      TRUE ~ Stress
    ),
    Drink_Amount = case_when(
        Drink_Amount == "Average Daily Drink Count: 1 or 2" ~ "1-2 Per Day",
        Drink_Amount == "Average Daily Drink Count: 3 or 4" ~ "3-4 Per Day",
        Drink_Amount %in% c("Average Daily Drink Count: 5 or 6", "Average Daily Drink Count: 7 to 9") ~ "5-9 Per Day",
        Drink_Amount == "Average Daily Drink Count: 10 or More" ~ ">=10 Per Day",
        TRUE ~ NA
    )
  )

In [ ]:
#-- Person and phenotype dataframe
person_pheno <- person_df %>%
  full_join(final, by = "person_id")

In [ ]:
library(tidyverse)
library(bigrquery)

#-- Extract conditions
condition_25694592_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20251014",  # Comment out this line if you want the export to always overwrite.
  "condition_25694592",
  "condition_25694592_*.csv")
message(str_glue('The data will be written to {condition_25694592_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_25694592_condition_df <- read_bq_export_from_workspace_bucket(condition_25694592_path)

In [ ]:
mdd_expanded_snomed <- c(
  # Core concepts
  "370143000",        # Major depressive disorder
  "192366006",        # Single major depressive episode
  "268620009",        # Single major depressive episode
  "268621008",        # Recurrent major depressive episodes
  "36923009",         # Major depression, single episode
  "66344007",         # Recurrent major depression
  "507521000000101",  # Major depressive disorder
  
  # Single major depressive episode – unspecified / mild / moderate / severe
  "191600009",        # Single major depressive episode, unspecified
  "191601008",        # Single major depressive episode, mild
  "191602001",        # Single major depressive episode, moderate
  "191603006",        # Single major depressive episode, severe, without mention of psychosis
  "191607007",        # Single major depressive episode NOS
  
  # Recurrent major depressive episodes – unspecified + by severity
  "191609005",        # Recurrent major depressive episodes, unspecified
  "191610000",        # Recurrent major depressive episodes, mild
  "191611001",        # Recurrent major depressive episodes, moderate
  "191612008",        # Recurrent major depressive episodes, severe, without mention of psychosis
  
  # Chronic / recurrent MDD
  "14183003",         # Chronic major depressive disorder, single episode
  "2618002",          # Chronic recurrent major depressive disorder
  
  # Severity + MDD wording
  "15639000",         # Moderate major depression, single episode
  "18818009",         # Moderate recurrent major depression
  "40379007",         # Mild recurrent major depression
  "450714000",        # Severe major depression
  "719592004",        # Moderately severe major depression
  "720451004",        # Minimal recurrent major depression
  "720452006",        # Moderately severe recurrent major depression
  "720453001",        # Moderately severe major depression single episode
  "720454007",        # Minimal major depression single episode
  "720455008",        # Minimal major depression
  "75084000",         # Severe major depression without psychotic features
  "76441001",         # Severe major depression, single episode, without psychotic features :contentReference[oaicite:3]{index=3}
  
  # ‘Major depressive disorder’ and ‘major depression’ variants
  "507531000000104",  # Mild major depression
  "507541000000108",  # Moderate major depression
  "507851000000109",  # Severe major depression without psychotic features
  "509781000000103",  # Severe major depression without psychotic features
  "510371000000107",  # Mild major depression
  "510381000000109",  # Moderate major depression
  "832007",           # Moderate major depression
  "87512008",         # Mild major depression
  
  # Melancholic / catatonic / atypical MDD
  "319768000",        # Recurrent MDD with melancholic features
  "320751009",        # Major depression, melancholic type
  "62951006",         # Major depression, melancholic type
  "38694004",         # Recurrent MDD with atypical features
  "39809009",         # Recurrent MDD with catatonic features
  "63778009",         # MDD, single episode with melancholic features
  "69392006",         # MDD, single episode with catatonic features
  "42925002",         # MDD, single episode with atypical features
  
  # “Major depression” with recurrence/severity
  "281000119103",     # Severe recurrent major depression
  "36474008",         # Severe recurrent major depression without psychotic features
  "606921000000104",  # Recurrent major depressive episodes, unspecified
  
  # Explicit “episode NOS” style codes
  "633681000000107",  # Single major depressive episode, unspecified
  "633701000000109",  # Single major depressive episode NOS
  "633721000000100",  # Recurrent major depressive episode NOS
  
  # Co-occurring anxiety but clearly MDD
  "16264621000119109",# Recurrent mild MDD co-occurrent with anxiety
  "16264821000119108",# Recurrent severe MDD co-occurrent with anxiety
  "16264901000119109",# Recurrent moderate MDD co-occurrent with anxiety
  "16265951000119109",# Mild MDD co-occurrent with anxiety single episode
  "16266831000119100",# Moderate MDD co-occurrent with anxiety single episode
  "16266991000119108" # Severe MDD co-occurrent with anxiety single episode
)


In [ ]:
#-- 
conditions <- dataset_25694592_condition_df

mdd <- conditions %>%
  filter(standard_concept_code %in% mdd_expanded_snomed)
mdd_cnt <- mdd %>%
  group_by(person_id) %>%
  summarise(
    mdd_count = n()
  )

anx <- conditions %>%
  filter(standard_concept_code %in% c("197480006"))
anx_cnt <- anx %>%
  group_by(person_id) %>%
  summarise(
    anx_count = n()
  )

#-- Map conditions back to person_pheno df
person_pheno_conditions <- person_pheno %>%
  mutate(
    Depression = ifelse(person_id %in% mdd_cnt$person_id, 1, 0),
    Anxiety = ifelse(person_id %in% anx_cnt$person_id, 1, 0)
  )


In [ ]:
#-- Migraine and Insomnia
condition_40757099_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260223",  # Comment out this line if you want the export to always overwrite.
  "condition_40757099",
  "condition_40757099_*.csv")
message(str_glue('The data will be written to {condition_40757099_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_40757099_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_40757099_condition_df <- read_bq_export_from_workspace_bucket(condition_40757099_path)

In [ ]:
mig_ins <- dataset_40757099_condition_df

mig <- mig_ins %>%
  filter(standard_concept_code != "193462001")
mig_cnt <- mig %>%
  group_by(person_id) %>%
  summarise(
    mig_count = n()
  )

ins <- mig_ins %>%
  filter(standard_concept_code == "193462001")
ins_cnt <- ins %>%
  group_by(person_id) %>%
  summarise(
    ins_count = n()
  )

#-- Map conditions back to person_pheno df
person_pheno_conditions <- person_pheno_conditions %>%
  mutate(
    Insomnia = ifelse(person_id %in% ins_cnt$person_id, 1, 0),
    Migraine = ifelse(person_id %in% mig_cnt$person_id, 1, 0)
  )


In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_44171489_path}` to copy these files
#       to the Jupyter disk.

measurement_44171489_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "20260223",  # Comment out this line if you want the export to always overwrite.
  "measurement_44171489",
  "measurement_44171489_*.csv")
message(str_glue('The data will be written to {measurement_44171489_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), measurement_type_concept_name = col_character(), operator_concept_name = col_character(), value_as_concept_name = col_character(), unit_concept_name = col_character(), visit_occurrence_concept_name = col_character(), measurement_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), unit_source_value = col_character(), value_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_44171489_measurement_df <- read_bq_export_from_workspace_bucket(measurement_44171489_path)

In [ ]:
BMI <- dataset_44171489_measurement_df %>%
    group_by(person_id) %>%
    summarise(BMI = median(value_as_number, na.rm = TRUE)) %>%
    ungroup()

person_pheno_conditions <- person_pheno_conditions %>%
    full_join(BMI, by = "person_id")

In [ ]:
# Save locally first
write.csv(person_pheno_conditions, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv", row.names = FALSE)

# Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp /home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/Phenotypes.csv")
))